# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Running in Colab')
    import nltk
    nltk.download('stopwords', quiet=True)


# Topic Modeling (LDA) In-Depth

Topic modeling is an unsupervised machine learning technique used to discover abstract "topics" that occur in a collection of documents. This notebook demonstrates how to use **Latent Dirichlet Allocation (LDA)** with Scikit-Learn to extract themes from a synthetic dataset of research abstracts.

## Objectives
1.  Generate a synthetic dataset with distinct underlying themes.
2.  Preprocess text data (Tokenization, Stopword Removal).
3.  Vectorize text using CountVectorizer (Bag of Words).
4.  Fit an LDA model.
5.  Visualize and interpret the topics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from nltk.corpus import stopwords
import re

# Ensure stopwords are downloaded locally if not in Colab
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

sns.set_style('whitegrid')

## 1. Generate Synthetic Data
We will create 200 documents belonging to 3 distinct categories:
*   **Technology**: Computers, AI, software, internet.
*   **Health**: Doctors, hospitals, medicine, patients.
*   **Environment**: Climate, nature, pollution, energy.

This ground truth helps us validate if the model correctly identifies these topics.

In [ ]:
np.random.seed(42)

tech_vocab = ['computer', 'software', 'internet', 'ai', 'data', 'algorithm', 'network', 'cyber', 'code', 'digital']
health_vocab = ['doctor', 'hospital', 'patient', 'medicine', 'nurse', 'surgery', 'health', 'clinic', 'treatment', 'virus']
env_vocab = ['climate', 'change', 'pollution', 'energy', 'solar', 'wind', 'nature', 'forest', 'ocean', 'carbon']

def generate_doc(vocab_list):
    # Pick 5-10 random words from the vocab
    doc_length = np.random.randint(5, 11)
    words = np.random.choice(vocab_list, doc_length, replace=True)
    # Add some random noise words
    noise = ['the', 'is', 'a', 'to', 'of', 'in', 'and']
    words = np.concatenate([words, np.random.choice(noise, 3)])
    np.random.shuffle(words)
    return ' '.join(words)

documents = []
labels = []

for _ in range(70):
    documents.append(generate_doc(tech_vocab))
    labels.append('Tech')

for _ in range(70):
    documents.append(generate_doc(health_vocab))
    labels.append('Health')

for _ in range(60):
    documents.append(generate_doc(env_vocab))
    labels.append('Environment')

df = pd.DataFrame({'text': documents, 'true_label': labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.head())

## 2. Text Preprocessing
We use a simple pipeline to clean the text and remove standard English stopwords.

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['clean_text'] = df['text'].apply(clean_text)
print(df[['text', 'clean_text']].head())

## 3. Vectorization (Bag of Words)
LDA requires a Document-Term Matrix (DTM). We use `CountVectorizer` to create this matrix, where rows are documents and columns are words (counts).

In [ ]:
vectorizer = CountVectorizer(max_df=0.95, min_df=2)
dtm = vectorizer.fit_transform(df['clean_text'])
feature_names = vectorizer.get_feature_names_out()

print(f"Document-Term Matrix Shape: {dtm.shape}")

## 4. Fit LDA Model
We specify `n_components=3` because we know there are 3 underlying topics. In real research, you might try multiple values and evaluate Coherence scores.

In [ ]:
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(dtm)

# Transform data to get topic distribution per document
topic_results = lda.transform(dtm)
df['predicted_topic'] = topic_results.argmax(axis=1)

print("Model Fitted.")

## 5. Visualize Topics
We extract the top 10 words for each topic to interpret what they represent.

In [ ]:
def plot_top_words(model, feature_names, n_top_words, title):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
    axes = axes.flatten()
    
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f'Topic {topic_idx + 1}', fontdict={'fontsize': 16})
        ax.invert_yaxis()
        ax.tick_params(axis='both', which='major', labelsize=10)
        for i in 'top right left'.split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=20)

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.show()

plot_top_words(lda, feature_names, 10, 'Top Words per Topic')

## 6. Evaluation
Since we have ground truth labels, let's see how well the unsupervised model separated the documents. Note that the Topic numbers (0, 1, 2) may not align directly with our Label names, but the grouping should be consistent.

In [ ]:
ct = pd.crosstab(df['true_label'], df['predicted_topic'])
print(ct)

plt.figure(figsize=(8, 6))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues')
plt.title('True Label vs Predicted Topic Cluster')
plt.ylabel('True Label')
plt.xlabel('Predicted Topic Cluster')
plt.show()